# HSTU vs SASRec — A100 long-context scaling

Runs cached one-event latency at **1k, 2k, 5k, 10k** history, batch=1,
plus KV-cache memory.

The embedded benchmark is unpacked to a real Python source file before import
so Triton 3.6 can inspect `@triton.jit` source normally.

In [ ]:
import base64, gzip, importlib.util, sys
from pathlib import Path

payload = 'H4sIAGuUeGoC/+0ca3PbuPG7fgVGN9OhHJmRlDi9uFHnfLZzzpzj+GLF14zr4UAiZLGiSJovS+n0v3d3AYIgRUtycr22M+exLQrYXSx2F/sAQU7jcMHeXa7SWRjYrpdEPl8xbxGFccrU11arpRoWPJ11WeotRGuKeC5P+cTnSSISjVM0aaQgW0QrxhMWREVTxAMXGuA3cou2NIwns8oXOwgILai32tMsmKReGHAfAd62Wien1++OT9lQgbgi9ybCak8yl7eZN1XN+NX2Eofn3PP52BdWhwk/Eaw9ibJ2p4UTgWEkMTtdRYINh0xS6bL2J4Dk7KfLT+w4BHQWZwGKwm63WpL+mE/mInATORAIa5H5Nvf98MFJpy8GwN4ozkQDMEypASxeHbYY/Ej4RKTO1A95+mLgSNJOFIuJl4AcrPbMu5vBDMRyIqKUndIHdEgCEakjir0gtdpE7RDmI8k6Ti5ipOE4nQLk+NPJUQmh+mlSGgSkUELQdO+APyl3J+ALYfU6nVY5h0KDsZeGwXqL7fPgLuN3AvWZ+gRwdnTljD6+G324KCRCc5HjjwiNWJAEKvOoiwGpisN1qm85aH+dLMsCbSI4RCyi2BI4n+Ojy6Pjd6PPgNvvOb1ev3X27mr04eO70ytosvrQ1Ouygfw4kB8E2Ou0fjwaHZ8hYuvXo4/vP13C5Yte693o9CPiDnq91sno82VpxErZsPpoPbG3by8sMJT3oZv5oiNn44opcxwv8FLHsRLhT7vM7TIvCESsIPAnySIRWx1bQ3bKLsCxA5HCqED7StxnAoya+5aGwB/oOvcCwWNLU+/WAX46Pf9krTcrPEIC5gyAjp7BNIwfeOyqCSwNzmORZnGgubSWHS2Oq6OrH/1wMt9JJsNXL7tsJribDAdqBsPBwaunyKgvRXTOVyK+COOF5dYgeJoGEuZ95qcejnaUpihOWKAVsbiKF2AsDqMwS4c9GwxlzNPJzJl6cZIOtb2XkipZGWxjZTpFTtBgtL5MuV2JOPeCu10kFzheKhbJsD9w+r1+l014xCdeuhoWK+EpIpS0kHl5Ve0uSEN/cVkFcJ1F6Aof+l+9rPYgNSmT08VYuC5NTo32jAHbqP6IU4fjucthr8ZaFCZ1/IIHRK5Bj9HuFIIU4LmXpNaNNskOmjRzQPQsBscmrEHnti4M0FtdjThQq/VDGUK1xo75ZKYc2PyQ+TAaXefGtS+Cu3R2CGOmQARVueBz4SQ8cSaIbZHwwPIAIYxXSm/zLsuBjZvbLvxRC3JOE0TuCUfNt1Q0ejFqI5u3IcA7ZM8a4GQGEBK3UNrz5+xM9wNiFEHws6SnAxm5tSVCzrLL4FeSKbVxMqv6GBlzhjJow4rCsD0kT2q4GrbHevagVEH+32RAObVCrRbqoNSLUl4Uh/8Qk9S5n+fWYsa77IvS2AmI9oudzHgkbvb7Ume/gvNAic8gvQkcRHUeBGQEabfSNva41NE9egfbl675S5f9enN4AhYwxg/J43wN4uRwsCeB5JWEy9fgoO+QwOhCQp0p5qqWQlZyUlqGkkuph3s798SDBbP1vmBCQeo4mRkxZL4VIt8IQetN2oAXTEUsAkhfUOHgu1ALuHqSVETF4qGVBC4VPAv4EaUR6Tyoy5arkJqXeg0guFXggEOSrYBmSy3dwOWtnQUJRF9BXBKBcyCAtMGDtfTS9L1uuTwFCFTEPFWLWy1UwyV/0Ss16GPw1JLtFivftLNyTaO5aWA5s/mN793eHHYZ/AJX8HlrT8Jo5VjzOmj+KGjeaWlYTqaTTLgvXMcNU7RRNwNWeHPUvDckNKglGussHp7jsE1Q+RYoFZOdiKJypctDX5ol3B9S5misb7vkzI4FrU5LuZCKHzTmj/axBOUanpSGBYVYvPMIGER2paZgAApV5NTKkSNhZDETpbOr0ScV8XdPmJwiWqg85R7Mxc3L+P+EsF+GbXVV7aYRoLMaQSTmPboh+F9rRrt18ydmDyLCUfpi/1WpgjRMOTI2APcMVPckEyBsaoDR95rYynLiC8R4yWModVKYvvQhYhGlK0sLj8h3SnEABoqINMR9KWuiBkYieCCzwCR18WJgWMp3jAKFC8OTGlkY+CsWCAG8pjMBoZ/HAXTnsJDDGNQVhCnjAbtYXjD0+Vi3x97Srk4jTZyHTdPoD1531tOkzTgoN62JfdbfNnnkoXnyG9GIjy1CI8BQpViqeNEq1hbeuEx2WSE5jz0epGWG/OL1weDb8mNFElhWV1+dHx/0tmXeut+bMj0u7nWEsWgfVpyej/lpUncFuGq6mFofYKVbjif8GkWfx3c7k/y+i1QHB/hnkExEFT/mXiLYNfczcRrHYWypAevq31oXIPNfXRcc9HasCyq8r7lj5OFRN1tBrRUVUoolyG1j9YDj7VA+4E5WkvJFlByq/YeRCJIw3lRbzJI0+82Li51rAxkGleTqSbrsRHn+jsXCLhzl38BQSwaDPpuFGchRpA9CQBKYQ7aU2DUt4naUptB3XvVoFwr/dOszpWguzanOcCNrEsMPgzvk6sUrRa5TzV7NpETbn6pzShbXah6yJ539OTSWzHa6lLE6Kv3OiwuTlrRR8MdimTppkQRjDm3k5HI9n2Nokkz+DF8V3UpKSJ3X6IqbOqk3QV+pd+uEFyTZwmqPZ253PAvc/b/C/zbx/bMSSKCBlcTPa0Lu3FyEgdDDx8JHp4T+BBiWlYBK/ioxtpgZ1Xc64acYeaNoKJZBwSmm3UpGOCkc8RbIlKKkqZ5LFsbZZC7SqjEVJnCn1gMfJxYR7tjgcRZQK3nBsN+R+5dWp8Oeg/W+6PWVoaDtWAUobo8Ovu9o261OAbODG8WCmsGDLBo8P7Ok/EEkeurPShqmQfaJh6rgTAutKzDQCnRBgZBnXG8rE8lwsU5UNvsV1aJZ7FGtksVOWkJW1YP1ojQpqUgEk/DP5JrcWIDu0X0cqFji1KoWJ7vVpt9QiyrVVVzgW5sCmSPrFggVFZ4gdkDmPpTag6tqRPxBqRWT6Kbt0sj3MJ+qRuDCE4MgTI/9tTBUJnw90K2+ytCzBQIM7l5+zPFDOw2aDO6x0AW4Dm8x3O8bKU8GoJnc7mgKSMWUjMCm6NPnUxDvFeL9Toj3xgZBMaX5zpi7b0MgySdsRUjwynZEYwBaV6dWkMFS1xywadvBjFXnOkx1G6xWbowYi6JCjdf3FwzBaRsDZVV5uGmGqnFaWWlNrC21bw6trM4IGCOQ5uhFlpUU4K2q3nAHblnEBdAH3otRVmynIXRRjoHOFmqI8kaddCE/qBt9//DSsiaTCpukXi4Mvc0FlMPGLaxfIAaDFy8n9PPpZ2d0BcHv9G8jvNAdlx+unF+7bIT/deOHT6Pyy9Hx6N31qXNRtiTAmCuce8iPU8ifwgACwTKK1wHcLQDzbRTmwTaAbUPk24bItw2RbxsiTbaRCLcxET46xtljHUW57TwGcPLLoz3Xj/X8eP7h+GfnYnP344RVfyN5Iy7OICijk4eoG4d3MccgbRmlpTLvQMLA4nGtwghp2UAj1GUvjIJFphgaTye7tGzuJR2VgkICVkzDcO/GUBX/8AssbWJ3r7R6aAKaZYNbdSkLnsyHAPAGNNBlYToTcXVftZiDut1teGTaEWhi9doUzkQCfRFxmFg3BcStLlgMynpPSWdNxSyVI3icl+/YR3nag/lhGB2yhzCes7s4fEjYg5fOSrdAe2/aGsstN8yXwP3HKeZKMISeU6GmYnYXnepeB+pdIj5rEsZFNSHKwcu6VDG80ZRbFYj5/DHtkmdca6lrfD5rAAnKUqIEDBoA3Xtd4xiQNZvRdkOzKUn/iVkmPppUZx2zwcY221mlmPNtrALmmK6VQ4GWll4y7NeQYtpsM4ozrfeyLtNrcrxJ7BR3gAJQ3CSK32iyc7Gqmv+6GVCARMWWWgLH/lW8PYWzokAFgAWIfAGqQJuHCrNYs/uKe8gf+navKmNZKmKxShO7U2VpUYA+4i0rmDiwFxQDFzzI7i7D24e6XtX1sKnZkdRjgWCIp1TVZhFgmVSUtmA0sOjHlX7Mnq0voBY0Ve9uEUK0+EIVrra+NXgAfYDBhaVYgdQV+agOnOebLOJ6q2PId3UMeaNjyBscQ/4Ex5BXHMP1b+UYMMA8037hwZxOniu/YEoSAaFuEFUJQgq5Jq+QQmduNNRmC0M3xdKc5mdm5827Z5Quf+P2WXnSo7wPFmZpuetFd33KsxJ6k3PT7mElHpvZDfWmtO1s3UiGbjdSUitZblVRZiM3Tx5mni+o5U1ZSpZhFTv2hmyg8PI1vLzEy020XKJRy8ba48Y663ZujZMMN71brQG6zsvr0nbKTR4EWSsRjY29rrFFVgKAaioEdW5RYcSW9kaHIczvffNghWbPhF5v7Te2mrfn80ZKeSOlfCOlingMYrXpm8NUW8wJng3PGqqHYXWX0Cwfhg23FE6uh+vb+iovwyOHa6XCEExvvUCAVgMfD8k88BhK8ZfFIRWjloYZob+6L2vlHTcmpTv4Y2fyj53JP3Ym//93JuVq/mNr8n9ia1KuggUPMu47iRCu1R+8kJyBeCdzfS7DPGMij1oQdXX3zxaQ1lomHqnHidBR1G+4G4TVAwfriGAlalB5A1bL6mZpT3zIYtVB5SV6yeqI9tyQ/w7gprpqXaV1FVQeBVVJqBH11AmOamrY/3M9LVSrZSVlVb8dV5FVdUgZAzsKWwpsLWg+ig/9JgHjaZhkFUxmcRigXSnmoJh0oJKFAaSZWcTtvhy2Q0VuBytOvDkaZZY6OjUJEy8QGumtLRucxFt4Pgf+VkSnNF0ip2++ygCraMmHXP7ZVpy0DwueunjwB8lCk7z4l8rO5RNRBetv8PzcC8YDt+Drr1BIvsYf8yma8w8XP+0ff7gYnf5txNTDNpdH8Pn5ED6vrtrFox8+26iamqjrMqbiQy2HogjCDmcMadDMmgZQ6PJ4kUVD+cANqSpOhvTIjQr/tSM1Et5IDaaBtYtuYeEmdNKlVXhJA/4UD2hYIsDHiRxYDbD66fkOSXj8FOAauzQfg1t0wOBWXONkmZ4AjbXeP65Nx9hITIpzLyBqn0eJcJEhYY2LY6d0NCGyecLjmK9wcyGp5Kr/1NTaeFTPWaDJSctc2tgChln6gnZ00DNBgHIk4glGPF9g+nXQq4K/PtgI/vqgBv56M/jrAvxfypb0oxPOYtx8wGm8SkXigBR6v9ujE2rIZ/LQ6hkeaJQswRXg7rGXpgYU9HPW7w1e7u0NzK2CHadWkv8et8FLb77rlEuOq4G+JGyVqeizMkvYYS7rgffloNNK6AiJ8ZxTU5BV+727RWQ6w1iHlgcbm8BbVNBINrZHLgJGjewW5+ieA7maFgq/kCKIXz+KeGg647//PZAdn9vmgSt5UwOE9FFMZKApkozyiSG4MlDIC1I9p12szxdjlx+Wj0kQxkQVlvBFBR7gGRBNd4CWAmuxLRloG+tUPRdLnSeXR2aXYgW61JXRJ635/Y/tw+q6rcyhhN/bS4vFrhjU3g6uzXCpv1K42hKDpFTRTBgaU1WwZvYGnTXR0iN7ZYqmZbuejRBuIWTsNk5ZV7JTOjm7kaJMjx4j2KjzabBNpzj9fVpLzWqVD9nuy82p/SgWU2/ZbuCctS9XI+TwK2yg6uGq4v49rYCcxKNmQL1fawcK+Tc0hMcpfoslSD/5v2IKNZH/x23BneL2mWuf8JS/xUcmEB3SJPVuBcudYuoaeXmI0nWnNl1aXuCK5VBPEZQS+tkiSIZKuPgAgp8J+K6yJnrsSKQOIeLAROemXIxOEkGAzCInTxzldW+1pSlg3f6c1dHbt61OhSYJ8huISrNAqloW1A3fF1TzfYUstNbr0ihGWGCppt5hQDy23wWQQ4PhpZzel8DUhKCw6dPzJfIAPfMSNuWwVGKWznig4uZfWFsSeaOBZUcJbrdxgq3v2IdIvbAiwtcUQFcwEXb5fgR6ncZdGN75Aio8fLtE8T6NGFaGtDa8shdhhlH9+SQMcDPoObWCCCAXmAgnFtQvH0/rlJQjns58TxO95GpvF+9LDelrneTz96sT+qTlE2VgQNGKzNxZ+P2FKuEA317MXS+2Ih7jMXSqVbpMLGGKTjg3ShdQZxo6kyS3cNDnsHyRMNgNRGioShze7/UcPBzsEB/L1AbYNj6+jto3JkRG8URahZ0+SlPmSlc8Fy7LoDfG9z4A8Y3vklBYJCfIOvDf3AOX4VZeGvFvIQcogGJFAAA='
source = gzip.decompress(base64.b64decode(payload)).decode("utf-8")

runtime = Path("/content/hstu_vs_sasrec_long_context_runtime.py")
runtime.write_text(source)
print("Runtime source:", runtime, "bytes:", runtime.stat().st_size)

spec = importlib.util.spec_from_file_location("longctx_bench", runtime)
mod = importlib.util.module_from_spec(spec)
sys.modules["longctx_bench"] = mod
spec.loader.exec_module(mod)